# 08D_Extract_Technical_Skills

Extract only technology-related skills from the LinkedIn skill universe for HSEP.

In [1]:
import pandas as pd
from rapidfuzz import process, fuzz

## Load Data

In [2]:
skills = pd.read_csv('../Generated Datasets/linkedin_skill_universe_1000.csv')
skills['skill'] = skills['skill'].astype(str).str.lower().str.strip()
print('Total Skills:', len(skills))

Total Skills: 2617


## Technical Taxonomy

In [3]:
TECH_TAXONOMY = {
    'Programming': [
        'python','java','javascript','typescript','c','c++','c#','go','rust','r',
        'php','kotlin','swift','programming','software development'
    ],

    'AI_ML': [
        'machine learning','deep learning','computer vision',
        'natural language processing','nlp','llm','rag','langchain',
        'tensorflow','pytorch','scikit-learn','artificial intelligence'
    ],

    'Data Analytics': [
        'data analysis','analytics','statistics','tableau','power bi',
        'powerbi','excel','data visualization','business intelligence'
    ],

    'Database': [
        'sql','mysql','postgresql','mongodb','oracle',
        'snowflake','nosql','database'
    ],

    'Cloud': [
        'aws','azure','gcp','google cloud','cloud computing'
    ],

    'DevOps': [
        'docker','kubernetes','jenkins','terraform',
        'gitlab','github actions','ci/cd','devops'
    ],

    'Cybersecurity': [
        'cybersecurity','network security','cloud security',
        'penetration testing','ethical hacking'
    ],

    'Software Engineering': [
        'software engineering','agile','agile development',
        'oop','system design'
    ],

    'Data Engineering': [
        'data engineering','etl','data warehouse','big data'
    ],

    'BI Tools': [
        'tableau','power bi','looker','qlik'
    ]
}

## Build Lookup Tables

In [4]:
lookup = {}
all_terms = []

for subcat, terms in TECH_TAXONOMY.items():
    for term in terms:
        lookup[term] = subcat
        all_terms.append(term)

## Technical Skill Extraction

In [5]:
results = []

for _, row in skills.iterrows():

    skill = row['skill']

    freq_col = [c for c in skills.columns if c != 'skill'][0]
    frequency = row[freq_col]

    found = False

    if skill in lookup:
        results.append([
            skill,
            frequency,
            lookup[skill],
            1.00
        ])
        continue

    for term, subcat in lookup.items():
        if term in skill or skill in term:
            results.append([
                skill,
                frequency,
                subcat,
                0.95
            ])
            found = True
            break

    if found:
        continue

    fuzzy = process.extractOne(
        skill,
        all_terms,
        scorer=fuzz.token_sort_ratio
    )

    if fuzzy and fuzzy[1] >= 90:
        matched = fuzzy[0]

        results.append([
            skill,
            frequency,
            lookup[matched],
            round(fuzzy[1]/100,2)
        ])

## Final Dataset

In [6]:
technical_df = pd.DataFrame(
    results,
    columns=[
        'skill',
        'linkedin_frequency',
        'sub_category',
        'confidence'
    ]
)

technical_df = technical_df.drop_duplicates(subset='skill')

technical_df = technical_df.sort_values(
    'linkedin_frequency',
    ascending=False
)

technical_df.to_csv(
    '../Generated Datasets/technical_skill_master.csv',
    index=False
)

print('Technical Skills Extracted:', len(technical_df))
technical_df.head()

Technical Skills Extracted: 2166


,skill,linkedin_frequency,sub_category,confidence
0,communication,370143,Programming,0.95
1,customer service,278102,Programming,0.95
2,teamwork,227609,Programming,0.95
3,communication skills,195949,Programming,0.95
4,leadership,185187,Programming,0.95
